In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
SEED = 42
np.random.seed(SEED)


In [2]:
DATA_PATH = "C:\\Users\\mnabielizzuddin.radz\\Downloads\\EG556N_Asssement\\data\\OGJ2014_extracted_vs_Table1.xlsx"
SHEET_ROWS = "OGJ2014_Extracted_Rows"

xls = pd.ExcelFile(DATA_PATH, engine="openpyxl")
print("Available sheets:", xls.sheet_names)

if SHEET_ROWS not in xls.sheet_names:
	SHEET_ROWS = xls.sheet_names[0]
	print(f"Sheet 'OGJ2014_Extracted_Rows' not found. Using sheet: {SHEET_ROWS}")

df = pd.read_excel(DATA_PATH, sheet_name=SHEET_ROWS, engine="openpyxl")
print(df.shape)
df.head(5)

Available sheets: ['OGJ2014_Extracted_Rows', 'OGJ2014_Agg_Ranges', 'Table1_Ranges_Input', 'Comparison_OGJ_vs_Table1']
(624, 15)


,technique,formation_category,porosity_min_pct,porosity_max_pct,perm_min_md,perm_max_md,depth_min_ft,depth_max_ft,api_min,api_max,visc_min_cp,visc_max_cp,so_start_min_pct,so_start_max_pct,raw_line
0,Steam,Sandstone,60.0,60.0,1.0,5.0,1000,1800,28.0,30.0,2.0,50.0,45.0,45.0,Aera Energy South Belridge Calif. Kern 1995 15...
1,Steam,Sandstone,32.0,32.0,200.0,2500.0,825,1650,12.0,13.0,2000.0,10000.0,60.0,60.0,Aera Energy Coalinga Calif. Fresno 1965 540 46...
2,Steam,Sandstone,34.0,34.0,800.0,1000.0,650,1000,9.0,10.0,11500.0,28000.0,55.0,55.0,Aera Energy Coalinga Calif. Fresno 1987 290 10...
3,Steam,Sandstone,36.0,36.0,1000.0,3000.0,1000,1000,11.0,14.0,1000.0,2000.0,65.0,65.0,Aera Energy Cymric Calif. Kern 1986 600 130 70...
4,Steam,Unconsolidated sands,40.0,40.0,1000.0,3000.0,200,200,13.0,13.0,1500.0,4000.0,60.0,60.0,Aera Energy Lost Hills Calif. Kern 1975 170 45...


In [3]:
df.columns.tolist() # Remove leading/trailing spaces from column names

['technique',
 'formation_category',
 'porosity_min_pct',
 'porosity_max_pct',
 'perm_min_md',
 'perm_max_md',
 'depth_min_ft',
 'depth_max_ft',
 'api_min',
 'api_max',
 'visc_min_cp',
 'visc_max_cp',
 'so_start_min_pct',
 'so_start_max_pct',
 'raw_line']

In [4]:
df["technique"].value_counts().head(20)

technique
Miscible CO2           204
Polymer                148
Miscible HC            120
Steam                  106
Combustion              24
Nitrogen immiscible     12
Hot water                4
HC immiscible            4
Miscible acid gas        2
Name: count, dtype: int64

In [5]:
df["formation_category"].value_counts(dropna=False)

formation_category
Sandstone               358
Carbonates              216
Unconsolidated sands     50
Name: count, dtype: int64

In [6]:
TABLE1_CLASSES = [
    "Steam",
    "Miscible CO2",
    "Miscible HC",
    "Polymer",
    "Combustion",
    "Surfactants",
    "Nitrates",
    "Microbial",
    "Hot water",
    "Miscible acid gas"
]

df = df[df["technique"].isin(TABLE1_CLASSES)].copy()
print(df.shape)
df["technique"].value_counts()

(608, 15)


technique
Miscible CO2         204
Polymer              148
Miscible HC          120
Steam                106
Combustion            24
Hot water              4
Miscible acid gas      2
Name: count, dtype: int64

In [7]:

def midpoint(a, b):
    return (a + b) / 2

def span(a, b):
    return (b - a)

# Map each variable to its min/max column names (based on your extracted file format) [1](https://petronas-my.sharepoint.com/personal/mnabielizzuddin_radz_petronas_com/_layouts/15/Doc.aspx?sourcedoc=%7BE3DF7B1F-87C2-4579-AA09-B5CD3DA51634%7D&file=Dataset_01_EOR.xlsx&action=default&mobileredirect=true)
pairs = {
    "depth_ft": ("depth_min_ft", "depth_max_ft"),
    "porosity_pct": ("porosity_min_pct", "porosity_max_pct"),
    "perm_md": ("perm_min_md", "perm_max_md"),
    "api": ("api_min", "api_max"),
    "visc_cp": ("visc_min_cp", "visc_max_cp"),
    "so_pct": ("so_start_min_pct", "so_start_max_pct"),
}

for base, (cmin, cmax) in pairs.items():
    df[f"{base}_mid"] = midpoint(df[cmin], df[cmax])
    df[f"{base}_span"] = span(df[cmin], df[cmax])

core_mid_cols = [f"{k}_mid" for k in pairs.keys()]
core_span_cols = [f"{k}_span" for k in pairs.keys()]

before = len(df)
df = df.dropna(subset=core_mid_cols + ["formation_category", "technique"]).copy()
after = len(df)

print(f"Dropped {before-after} rows due to missing core inputs. Remaining: {after}")

Dropped 91 rows due to missing core inputs. Remaining: 517


In [8]:
# Add small epsilon to avoid log(0)
EPS = 1e-6

df["log_perm_mid"] = np.log10(df["perm_md_mid"] + EPS)
df["log_visc_mid"] = np.log10(df["visc_cp_mid"] + EPS)

# You can also log spans if you want (optional)
df["log_perm_span"] = np.log10(df["perm_md_span"].clip(lower=0) + 1 + EPS)
df["log_visc_span"] = np.log10(df["visc_cp_span"].clip(lower=0) + 1 + EPS)

In [9]:
df = pd.get_dummies(df, columns=["formation_category"], prefix="form", drop_first=False)
df.filter(like="form_").head()

,form_Carbonates,form_Sandstone,form_Unconsolidated sands
0,False,True,False
1,False,True,False
2,False,True,False
3,False,True,False
4,False,False,True


In [10]:
# Base numeric features
numeric_features = core_mid_cols + core_span_cols + [
    "log_perm_mid", "log_visc_mid",
    "log_perm_span", "log_visc_span"
]
# Formation one-hots
formation_features = [c for c in df.columns if c.startswith("form_")]

FEATURES = numeric_features + formation_features

X = df[FEATURES].values
y_text = df["technique"].values

print("X shape:", X.shape)
print("y shape:", y_text.shape)
print("Num features:", len(FEATURES))
le = LabelEncoder()
y = le.fit_transform(y_text)

class_names = le.classes_
num_classes = len(class_names)

print("Classes:", class_names)
print("num_classes:", num_classes)


X shape: (517, 19)
y shape: (517,)
Num features: 19
Classes: ['Combustion' 'Hot water' 'Miscible CO2' 'Miscible HC' 'Miscible acid gas'
 'Polymer' 'Steam']
num_classes: 7


In [15]:
MIN_COUNT = 100

counts = df["technique"].value_counts()
trainable_classes = counts[counts >= MIN_COUNT].index.tolist()
rare_classes = counts[counts < MIN_COUNT].index.tolist()

print("Trainable classes:", trainable_classes)
print("Rare classes (fuzzy-only):", rare_classes)

df_trainable = df[df["technique"].isin(trainable_classes)].copy()
df_rare = df[df["technique"].isin(rare_classes)].copy()

print("Trainable rows:", df_trainable.shape)
print("Rare rows:", df_rare.shape)

Trainable classes: ['Miscible CO2', 'Polymer', 'Miscible HC']
Rare classes (fuzzy-only): ['Steam', 'Combustion', 'Hot water', 'Miscible acid gas']
Trainable rows: (395, 33)
Rare rows: (122, 33)


In [16]:
X = df_trainable[FEATURES].values
y_text = df_trainable["technique"].values

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y_text)

class_names = le.classes_
num_classes = len(class_names)

print("Classes used in NN:", class_names)
print("num_classes:", num_classes)

Classes used in NN: ['Miscible CO2' 'Miscible HC' 'Polymer']
num_classes: 3


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.10, random_state=SEED, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.20, random_state=SEED, stratify=y_temp
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

Train: (355, 19) Val: (32, 19) Test: (8, 19)


In [20]:
from sklearn.linear_model import LogisticRegression

baseline = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",   # helps imbalance
    #multi_class="multinomial",
    n_jobs=-1
)

baseline.fit(X_train, y_train)
pred = baseline.predict(X_test)

print(classification_report(y_test, pred, target_names=class_names))

c:\Users\mnabielizzuddin.radz\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


              precision    recall  f1-score   support

Miscible CO2       0.75      0.75      0.75         4
 Miscible HC       0.50      0.50      0.50         2
     Polymer       1.00      1.00      1.00         2

    accuracy                           0.75         8
   macro avg       0.75      0.75      0.75         8
weighted avg       0.75      0.75      0.75         8



c:\Users\mnabielizzuddin.radz\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
